# Phân loại review Olist bằng embedding

Khoảng 41% đơn Olist có review kèm lời bình bằng tiếng Bồ Đào Nha. Bài toán: từ **nội dung lời
bình**, dự đoán review có xấu không (`review_score <= 2`). Bạn tự viết tokenizer, vocabulary, một
model "túi embedding" (trung bình embedding các từ rồi qua một tầng Linear), so nó với baseline
TF-IDF + logistic regression, và xem embedding học được gì.

Dùng định nghĩa chung của track: đơn giao xong (`delivered`, có ngày giao), giữ review có
`review_answer_timestamp` mới nhất mỗi đơn, chia theo ngày mua (train trước 2018-01-01, validation
2018-01 tới 2018-03, test 2018-04 tới 2018-08).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và train; test bỏ qua.
- Chấm: `uv run pytest tracks/data-science/modules/13-deep-learning/exercises/intermediate/03-olist-review-embeddings -q`

In [ ]:
import collections
import copy
import re
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from torch import nn

TRAIN_END = pd.Timestamp("2018-01-01")
VAL_END = pd.Timestamp("2018-04-01")
TEST_END = pd.Timestamp("2018-09-01")

## 0. Dữ liệu (đã viết sẵn)

`load_reviews` trả một dòng mỗi đơn giao xong có lời bình không rỗng: `order_id`,
`order_purchase_timestamp`, `text`, `review_score`, `bad`.

In [ ]:
def load_reviews(raw_dir) -> pd.DataFrame:
    raw = Path(raw_dir)
    dates = ["order_purchase_timestamp", "order_delivered_customer_date"]
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=dates)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    df = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = df.merge(latest[["order_id", "review_score", "review_comment_message"]], on="order_id")
    df["text"] = df.review_comment_message.fillna("").str.strip()
    df = df[df.text.str.len() > 0].copy()
    df["bad"] = (df.review_score <= 2).astype(int)
    return df[["order_id", "order_purchase_timestamp", "text", "review_score", "bad"]].reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    t = df.order_purchase_timestamp
    return {"train": df[t < TRAIN_END], "val": df[(t >= TRAIN_END) & (t < VAL_END)],
            "test": df[(t >= VAL_END) & (t < TEST_END)]}

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
reviews = load_reviews(ROOT / "data" / "raw")
splits = time_split(reviews)
print(len(reviews), "review có lời bình; tỉ lệ xấu", round(reviews.bad.mean(), 4))
print({k: (len(v), round(v.bad.mean(), 4)) for k, v in splits.items()})
print(reviews.text.str.split().str.len().describe().round(1).to_dict())
reviews.sample(5, random_state=0)[["review_score", "text"]]

## 1. Tokenizer và vocabulary

- `tokenize(text)`: chữ thường, **bỏ dấu** (người Brazil hay gõ "nao" thay cho "não"; chuẩn hoá
  `unicodedata.normalize("NFKD", ...)` rồi bỏ ký tự có `unicodedata.combining(c)`), giữ các cụm
  `[a-z]+` (bỏ số, dấu câu, emoji).
- `build_vocab(texts, min_freq=2, max_size=None)`: `{"<pad>": 0, "<unk>": 1}` rồi các từ có tần
  suất ≥ `min_freq`, sắp theo tần suất giảm dần, hoà thì theo thứ tự chữ cái. `max_size` tính cả
  hai token đặc biệt.
- `encode(texts, vocab, max_len)`: `LongTensor (N, max_len)`, cắt bớt nếu dài, đệm 0 nếu ngắn, từ
  lạ thành 1.

In [ ]:
def tokenize(text: str) -> list[str]:
    # TODO
    raise NotImplementedError


def build_vocab(texts, min_freq: int = 2, max_size: int | None = None) -> dict[str, int]:
    # TODO
    raise NotImplementedError


def encode(texts, vocab: dict[str, int], max_len: int = 50) -> torch.Tensor:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kích thước vocabulary và tỉ lệ từ lạ trên validation, test
vocab = build_vocab(splits["train"].text, min_freq=2)
print("vocab", len(vocab))
for name in ("val", "test"):
    toks = [t for text in splits[name].text for t in tokenize(text)]
    print(name, "tỉ lệ <unk>", round(sum(t not in vocab for t in toks) / len(toks), 4))
X_tr, X_va, X_te = (encode(splits[k].text.tolist(), vocab, max_len=50) for k in ("train", "val", "test"))
y_tr, y_va, y_te = (torch.tensor(splits[k].bad.to_numpy(), dtype=torch.float32) for k in ("train", "val", "test"))

## 2. Model túi embedding

`BagOfEmbeddings(vocab_size, dim)` có đúng hai thuộc tính tham số:

- `self.embedding`: `nn.Embedding` hoặc `nn.EmbeddingBag` (`vocab_size × dim`);
- `self.out`: `nn.Linear(dim, 1)`.

`forward(ids)` với `ids (N, L)` trả logits `(N,)`: **trung bình embedding của các token khác 0**
(dòng toàn padding cho vector 0), rồi qua `self.out`. Padding không được ảnh hưởng tới kết quả.

In [ ]:
class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, dim: int = 32):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

## 3. Train có early stopping

`train_text_model(model, X, y, X_val, y_val, epochs=10, lr=1e-2, batch_size=256, patience=2, seed=0)`:
Adam, `BCEWithLogitsLoss`, xáo bằng `torch.Generator().manual_seed(seed + epoch)`, sau mỗi epoch đo
val loss (chế độ eval, không gradient), early stopping và khôi phục trọng số tốt nhất **giống hệt
quy tắc của bài Basic 02**. Trả `{"train_loss": [...], "val_loss": [...], "best_epoch": int}`.

In [ ]:
def train_text_model(model: nn.Module, X, y, X_val, y_val, epochs: int = 10, lr: float = 1e-2,
                     batch_size: int = 256, patience: int = 2, seed: int = 0) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: baseline TF-IDF + logistic regression (cùng tokenizer) và model túi embedding
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
A_tr = tfidf.fit_transform(splits["train"].text)
logreg = LogisticRegression(C=4, max_iter=2000).fit(A_tr, splits["train"].bad)
rows = {}
for name in ("val", "test"):
    p = logreg.predict_proba(tfidf.transform(splits[name].text))[:, 1]
    rows[("TF-IDF + LR", name)] = (roc_auc_score(splits[name].bad, p), average_precision_score(splits[name].bad, p))

torch.manual_seed(0)
bag = BagOfEmbeddings(len(vocab), dim=32)
start = time.perf_counter()
hist = train_text_model(bag, X_tr, y_tr, X_va, y_va, epochs=10, lr=1e-2, batch_size=256, patience=2, seed=0)
print("túi embedding: best_epoch", hist["best_epoch"], "val_loss", [round(v, 4) for v in hist["val_loss"]],
      f"{time.perf_counter() - start:.1f} s")
bag.eval()
with torch.no_grad():
    for name, X_, y_ in (("val", X_va, y_va), ("test", X_te, y_te)):
        p = torch.sigmoid(bag(X_)).numpy()
        rows[("Túi embedding", name)] = (roc_auc_score(y_, p), average_precision_score(y_, p))
pd.DataFrame(rows, index=["AUC", "PR-AUC"]).T.round(4)

## 4. Embedding học được gì?

`nearest_words(weight, vocab, word, k=5)`: k từ có **cosine similarity** cao nhất với `word` trong
ma trận `weight (V, dim)`, bỏ chính nó, `<pad>` và `<unk>`; từ không có trong vocab thì `KeyError`.

In [ ]:
def nearest_words(weight: torch.Tensor, vocab: dict[str, int], word: str, k: int = 5) -> list[str]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: hàng xóm gần nhất trong embedding của model phân loại, và các từ "kéo về xấu" mạnh nhất
W = bag.embedding.weight.detach()
for w in ["pessimo", "otimo", "atraso", "defeito"]:
    print(w, nearest_words(W, vocab, w))
score = (W @ bag.out.weight.detach().T).squeeze()      # đóng góp của từng từ vào logit
inverse = {i: t for t, i in vocab.items()}
print("kéo về xấu:", [inverse[i] for i in score.topk(10).indices.tolist()])
print("kéo về tốt:", [inverse[i] for i in (-score).topk(10).indices.tolist()])

## 5. Câu hỏi (viết câu trả lời vào cell này)

1. Model túi embedding có thắng TF-IDF + logistic regression không? Vì sao điều đó không đáng ngạc
   nhiên với dữ liệu cỡ này?
2. Hàng xóm gần nhất của "pessimo" trong embedding của model phân loại có hợp lý không? Vì sao
   hướng "kéo về xấu / tốt" lại rõ ràng hơn nhiều?
3. Tỉ lệ review xấu trong các review có lời bình là bao nhiêu, so với 12.8% trên mọi review? Điều
   đó giới hạn việc dùng model này cho bài toán "dự đoán review xấu của mọi đơn" ra sao?